# 2 · train_grid — ablation runs  *(v2: protocol-matched to run5/exp_005)*
| Run | Train data | Architecture |
|---|---|---|
| A | PKU-483 | stock v11n *(retrained in the unified space — old Run1 ckpt uses a different class order)* |
| B | PKU-483 +ModRand | stock |
| C | PKU-483 | IBN only |
| D | PKU-483 | Edge stem only |
| E | PKU-483 | IBN + Edge |
| F (full) | PKU-483 +ModRand | IBN + Edge |
| G | Kaggle +ModRand | IBN + Edge |
| H / I | DeepPCB / +ModRand | stock / IBN + Edge |
| J (oracle) | Kaggle + DeepPCB joint | stock — registered from run5's exp_005 |

All runs use **exp_005's exact protocol** (200 ep, seed 42, explicit strong aug) so row J is directly comparable. **Run the sanity cell before burning GPU-days.**

In [ ]:
# --- project root resolution (same pattern as run1-run5) ---
import os, sys, json
from pathlib import Path

def resolve_project_root() -> Path:
    env = os.environ.get("PCB_PROJECT_ROOT")
    if env:
        return Path(env).resolve()
    cwd = Path.cwd().resolve()
    if cwd.name == "experiments":
        return cwd.parent
    if (cwd / "experiments").is_dir():
        return cwd
    return cwd

PROJECT_ROOT = resolve_project_root()
EXP_DIR = PROJECT_ROOT / "experiments"
sys.path.insert(0, str(EXP_DIR))   # mr_yolo11.py + make_modrand_dataset.py live here
print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
import mr_yolo11
from mr_yolo11 import MRTrainer, IBNOnlyTrainer, EdgeOnlyTrainer
from ultralytics import YOLO

YAML_DIR = EXP_DIR / "yamls"
RUNS_DIR = PROJECT_ROOT / "runs_grid"
REGISTRY = EXP_DIR / "runs_registry.json"

# ---- protocol-matched to run5/exp_005: ONLY the grid variable changes ----
EPOCHS, IMGSZ, BATCH, SEED, DEVICE = 200, 640, 16, 42, 0
AUG = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.5,
           degrees=10.0, translate=0.10, scale=0.5, shear=2.0,
           flipud=0.5, fliplr=0.5,
           mosaic=1.0, mixup=0.10, copy_paste=0.0)
BASE = dict(epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, seed=SEED, device=DEVICE,
            deterministic=True, patience=50,
            project=str(RUNS_DIR), exist_ok=True, **AUG)

# Optional W&B: rotate your key first, login via CLI, never hardcode it
USE_WANDB = False
if USE_WANDB:
    from ultralytics import settings
    settings.update({"wandb": True})

In [ ]:
def register(run_id, best, meta):
    reg = json.loads(REGISTRY.read_text()) if REGISTRY.exists() else {}
    reg[run_id] = {"best": str(best), **meta}
    REGISTRY.write_text(json.dumps(reg, indent=2))
    print(f"registered {run_id}: {best}")

def train_run(run_id, data_yaml, trainer=None, **overrides):
    args = {**BASE, **overrides, "data": str(YAML_DIR / data_yaml), "name": run_id}
    model = YOLO("yolo11n.pt")
    if trainer:
        model.train(trainer=trainer, **args)
    else:
        model.train(**args)
    best = Path(model.trainer.best)
    register(run_id, best,
             {"data": data_yaml, "trainer": trainer.__name__ if trainer else "stock"})
    return best

In [ ]:
# J: the joint-training ORACLE from run5 (already in the unified space)
JOINT_BEST = PROJECT_ROOT / "results/exp_005_yolov11n_joint_kaggle_deeppcb/weights/best.pt"
if JOINT_BEST.exists():
    register("J_joint_oracle", JOINT_BEST, {"data": "joint(kaggle+deeppcb)", "trainer": "stock"})
else:
    print("exp_005 not finished yet — rerun this cell when it is:", JOINT_BEST)

# Old Run1's checkpoint predicts in Run1's ORIGINAL class order -> row A is
# retrained below instead (index-consistent by construction).
# Run2's order == unified (run5 asserted it), so it can be registered as-is:
RUN2_BEST = PROJECT_ROOT / "results/exp_002_yolov11n_kaggle/weights/best.pt"   # EDIT name
if RUN2_BEST.exists():
    register("Run2_kaggle_stock", RUN2_BEST, {"data": "kaggle.yaml", "trainer": "stock"})
else:
    print("EDIT RUN2_BEST if you want the old Run2 row:", RUN2_BEST)

In [ ]:
# SANITY (10 epochs, ~10 min): full method end-to-end before the real grid.
# Then open eval_matrix.ipynb and confirm "F_sanity" loads and validates.
train_run("F_sanity", "pku_modrand.yaml", MRTrainer, epochs=10)

In [ ]:
train_run("A_pku_stock", "pku.yaml")

In [ ]:
train_run("B_pku_modrand_stock", "pku_modrand.yaml")

In [ ]:
train_run("C_pku_ibn", "pku.yaml", IBNOnlyTrainer)

In [ ]:
train_run("D_pku_edge", "pku.yaml", EdgeOnlyTrainer)

In [ ]:
train_run("E_pku_mr_arch", "pku.yaml", MRTrainer)

In [ ]:
train_run("F_pku_full", "pku_modrand.yaml", MRTrainer)

In [ ]:
train_run("G_kaggle_full", "kaggle_modrand.yaml", MRTrainer)

In [ ]:
train_run("H_dpcb_stock", "deeppcb.yaml")

In [ ]:
train_run("I_dpcb_full", "deeppcb_modrand.yaml", MRTrainer)

In [ ]:
# registry overview (remove F_sanity once the real F is trained)
reg = json.loads(REGISTRY.read_text())
for k, v in reg.items():
    print(f"{k:24s} {v['trainer']:18s} {v['data']:24s} {v['best']}")